In [ ]:
# ============================================
# INSTALL LIBRARIES
# ============================================
!pip install pandas nltk scikit-learn matplotlib seaborn

# ============================================
# IMPORT LIBRARIES
# ============================================
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score
from sklearn.decomposition import PCA

nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ============================================
# LOAD DATASET
# ============================================
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, encoding='latin-1')

df.columns = df.columns.str.strip().str.lower()

text_col = None
for col in df.columns:
    if 'response' in col or 'feedback' in col or 'text' in col:
        text_col = col
        break

# ============================================
# PREPROCESSING
# ============================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)

    words = [
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in stop_words and len(w) > 2
    ]
    return " ".join(words)

df['cleaned'] = df[text_col].apply(clean_text)

# ============================================
# LABELING
# ============================================
keywords = {
    'Appreciation': ['thank','grateful','appreciate','good','great','satisfied'],
    'Challenges': ['problem','issue','delay','error','slow','difficult'],
    'Suggestion': ['should','recommend','improve','better','suggest']
}

def assign_labels(text):
    labels = []
    for label, words in keywords.items():
        if any(w in text for w in words):
            labels.append(label)
    return labels if labels else ['Neutral']

df['labels'] = df['cleaned'].apply(assign_labels)

# ============================================
# FEATURES
# ============================================
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['cleaned'])

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])

# ============================================
# SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================
# MODELS
# ============================================
models = {
    "Decision Tree": OneVsRestClassifier(DecisionTreeClassifier()),
    "KNN": OneVsRestClassifier(KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": OneVsRestClassifier(MultinomialNB()),
    "Neural Network": OneVsRestClassifier(MLPClassifier(hidden_layer_sizes=(50,), max_iter=500)),
    "SVM": OneVsRestClassifier(SVC(kernel='linear', probability=True))
}

results = {}

# ============================================
# TRAIN ALL
# ============================================
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    f1_micro = f1_score(y_test, y_pred, average='micro')
    f1_macro = f1_score(y_test, y_pred, average='macro')

    results[name] = (f1_micro, f1_macro)

# ============================================
# RESULTS
# ============================================
print("\nMODEL PERFORMANCE")
for k, v in results.items():
    print(f"{k}: F1 Micro={v[0]:.4f}, F1 Macro={v[1]:.4f}")

# ============================================
# GRAPH
# ============================================
labels_plot = list(results.keys())
micro_scores = [v[0] for v in results.values()]
macro_scores = [v[1] for v in results.values()]

x = np.arange(len(labels_plot))

plt.figure(figsize=(10,5))
plt.bar(x - 0.2, micro_scores, 0.4, label='F1 Micro')
plt.bar(x + 0.2, macro_scores, 0.4, label='F1 Macro')

plt.xticks(x, labels_plot, rotation=30)
plt.title("Model Performance Comparison")
plt.legend()
plt.show()

# ============================================
# INTERACTIVE
# ============================================
def predict(text, model_choice):
    model = models[model_choice]

    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])

    pred = model.predict(vec)[0]

    labels = [mlb.classes_[i] for i, v in enumerate(pred) if v == 1]

    if not labels:
        labels = ["Neutral"]

    return labels

print("Models:", list(models.keys()))

while True:
    text = input("Enter text (or exit): ")
    if text.lower() == "exit":
        break

    print("Choose model:")
    for i, m in enumerate(models.keys()):
        print(i+1, m)

    idx = int(input("Model number: ")) - 1
    model_name = list(models.keys())[idx]

    labels = predict(text, model_name)

    print("Predicted Labels:", labels)